# Naive NL-to-SQL Baseline

This notebook reproduces the **single-pass baseline** described in the paper.
It loads `defog/llama-3-sqlcoder-8b`, prompts it with the schema and a natural-language question,
and saves the generated SQL without retries, decomposition, or tool use.

From a code-review perspective, this notebook is intentionally simple:
- deterministic generation (`temperature=0`)
- explicit schema grounding
- lightweight output export for evaluation
- minimal helper utilities for readability and reuse


In [ ]:
# Colab setup
# The notebook is designed for a GPU-backed Colab runtime.
!pip install transformers accelerate torch --index-url https://download.pytorch.org/whl/cu121


In [ ]:
import csv
import json
import os
import re
import textwrap
from typing import Dict, List

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:
# -----------------------------
# Configuration
# -----------------------------
MODEL_NAME = "defog/llama-3-sqlcoder-8b"

# Deterministic generation keeps the baseline reproducible and makes comparison to the
# agentic notebook more defensible in an experimental setting.
GEN_KW = {
    "max_new_tokens": 512,
    "temperature": 0.0,
    "do_sample": False,
}

assert torch.cuda.is_available(), (
    "Please enable a GPU in Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU"
)


In [ ]:
# -----------------------------
# Load model and tokenizer
# -----------------------------
# In the paper, this model acts as the atomic SQL generator.
# Here it is used directly, without orchestration or self-correction.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
    device_map="auto",
)

# Stop on either the model EOS token or Llama-3's end-of-turn token when available.
try:
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
except Exception:
    eot_id = None

eos_token_ids = [token_id for token_id in [tokenizer.eos_token_id, eot_id] if token_id is not None]


In [ ]:
# -----------------------------
# Schema and evaluation questions
# -----------------------------
DB_SCHEMA = """
CREATE TABLE public.checkins_nyc (
  user_id text,
  place_id text,
  category_id text,
  category_name text,
  latitude real,
  longitude real,
  checkin_time timestamp without time zone NOT NULL
);
CREATE TABLE public.checkins_tokyo (
  user_id text,
  place_id text,
  category_name text,
  latitude real,
  longitude real,
  checkin_time timestamp without time zone
);
""".strip()

QUESTIONS = [
    "Compare the most popular categories in NYC vs Tokyo.",
    "Which city has more late-night check-ins?",
    "Show the average check-ins per user in NYC vs Tokyo.",
    "Which city has more check-ins near train stations?",
    "Compare weekend activity patterns between NYC and Tokyo.",
]


In [ ]:
# -----------------------------
# Prompting utilities
# -----------------------------
def build_user_message(user_question: str, create_table_statements: str, instructions: str = "") -> str:
    """Create the SQLCoder prompt in the chat format expected by the model."""
    return textwrap.dedent(
        f"""        <|begin_of_text|><|start_header_id|>user<|end_header_id|>

        Generate a SQL query to answer this question: `{user_question}`
        {instructions}
        Follow the DDL statements strictly.

        DDL statements:
        {create_table_statements}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

        The following SQL query best answers the question `{user_question}`:
        ```sql
        """
    )


SQL_BLOCK_RE = re.compile(r"```sql\s*(.*?)```", re.IGNORECASE | re.DOTALL)


def extract_sql(text: str) -> str:
    """Extract SQL from the model response while staying tolerant to formatting drift."""
    match = SQL_BLOCK_RE.search(text)
    if match:
        return match.group(1).strip()

    fallback_index = text.lower().rfind("```sql")
    if fallback_index != -1:
        return text[fallback_index + len("```sql"):].strip().strip("`").strip()

    return text.strip()


In [ ]:
# -----------------------------
# Inference loop
# -----------------------------
@torch.no_grad()
def generate_sql_for_questions(questions: List[str]) -> List[Dict[str, str]]:
    """Run the baseline model once per question and collect the generated SQL."""
    rows: List[Dict[str, str]] = []

    for idx, question in enumerate(questions, start=1):
        user_message = build_user_message(question, DB_SCHEMA).strip()
        messages = [{"role": "user", "content": user_message}]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        output_ids = model.generate(
            **inputs,
            eos_token_id=eos_token_ids[0] if eos_token_ids else None,
            pad_token_id=tokenizer.eos_token_id,
            **GEN_KW,
        )

        generated_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
        response_text = tokenizer.decode(generated_tokens, skip_special_tokens=False)
        sql = extract_sql(response_text)

        rows.append({"id": idx, "question": question, "sql": sql})
        print(f"[{idx}/{len(questions)}] Generated SQL for: {question}")

    return rows


results = generate_sql_for_questions(QUESTIONS)


In [ ]:
# -----------------------------
# Save outputs for inspection or downstream evaluation
# -----------------------------
os.makedirs("/content", exist_ok=True)
csv_path = "/content/queries.csv"
jsonl_path = "/content/queries.jsonl"

with open(csv_path, "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["id", "question", "sql"])
    writer.writeheader()
    writer.writerows(results)

with open(jsonl_path, "w", encoding="utf-8") as jsonl_file:
    for row in results:
        jsonl_file.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved files:")
print(csv_path)
print(jsonl_path)


In [ ]:
# Quick preview
for row in results[:3]:
    print("\n---")
    print(f"Q{row['id']}: {row['question']}")
    print("SQL:")
    print(row["sql"])
